# Create MQ Mental Health Research Awards

Builds the awards table for **MQ Mental Health Research** (funder_id
`4320312944`, IAMHRF priority-40 core-MH, priority `271`) from their
research-projects WP REST + detail pages (scraped via Playwright — the site sits
behind a Cloudflare sgcaptcha that a headless-browser warmup clears; see
`scripts/local/mq_mental_health_to_s3.py`). 65 projects with title, PI (60/65,
labelled field), institution, country, award scheme, and funding period. No
amounts at source (§6.7 waiver).

Source parquet: `s3://openalex-ingest/awards/mq_mental_health/mq_mental_health_projects.parquet`

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.mq_mental_health_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/mq_mental_health/mq_mental_health_projects.parquet`;

In [ ]:
%sql
-- Check row count (should be ~65)
SELECT COUNT(*) as total_projects FROM openalex.awards.mq_mental_health_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.mq_mental_health_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.mq_mental_health_raw LIMIT 5;

## Step 2: Create MQ Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.mq_mental_health_awards
USING delta
AS
WITH
mq_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320312944  -- MQ Mental Health Research
),
awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000 as id,
        g.title as display_name,
        g.description as description,
        f.funder_id,
        g.funder_award_id as funder_award_id,
        CAST(NULL AS DECIMAL(18,2)) as amount,
        CAST(NULL AS STRING) as currency,
        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name, f.ror_id, f.doi
        ) as funder,
        'grant' as funding_type,
        g.scheme as funder_scheme,
        'mq_mental_health' as provenance,
        CAST(NULL AS DATE) as start_date,
        CAST(NULL AS DATE) as end_date,
        TRY_CAST(regexp_extract(g.funding_period_raw, '^(\\d{4})', 1) AS INT) as start_year,
        TRY_CAST(regexp_extract(g.funding_period_raw, '(\\d{4})$', 1) AS INT) as end_year,
        CASE
            WHEN g.pi_family IS NOT NULL THEN
                struct(
                    g.pi_given as given_name,
                    g.pi_family as family_name,
                    CAST(NULL AS STRING) as orcid,
                    CAST(NULL AS DATE) as role_start,
                    struct(
                        g.institution as name,
                        CASE g.location
                            WHEN 'UK' THEN 'United Kingdom'
                            WHEN 'USA' THEN 'United States'
                            ELSE g.location
                        END as country,
                        CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
                    ) as affiliation
                )
            ELSE NULL
        END as lead_investigator,
        CAST(NULL AS STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,
        g.landing_page_url,
        CAST(NULL AS STRING) as doi,
        CAST(NULL AS STRING) as works_api_url,
        current_timestamp() as created_date,
        current_timestamp() as updated_date
    FROM openalex.awards.mq_mental_health_raw g
    CROSS JOIN mq_funder f
)
SELECT * FROM awards_transformed;

## Step 3: Insert into openalex_awards_raw

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'mq_mental_health' AND priority = 271;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    271 as priority
FROM openalex.awards.mq_mental_health_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_awards FROM openalex.awards.mq_mental_health_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, start_year, end_year
FROM openalex.awards.mq_mental_health_awards LIMIT 10;

In [ ]:
%sql
-- should all be MQ
SELECT funder.display_name, COUNT(*) as cnt
FROM openalex.awards.mq_mental_health_awards GROUP BY 1 ORDER BY 2 DESC;

In [ ]:
%sql
SELECT funder_scheme, COUNT(*) as cnt
FROM openalex.awards.mq_mental_health_awards GROUP BY 1 ORDER BY 2 DESC;

In [ ]:
%sql
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(lead_investigator) as has_pi,
    COUNT(lead_investigator.affiliation.name) as has_institution,
    COUNT(start_year) as has_year,
    ROUND(try_divide(COUNT(lead_investigator), COUNT(*)) * 100.0, 1) as pct_with_pi
FROM openalex.awards.mq_mental_health_awards;

In [ ]:
%sql
SELECT lead_investigator.affiliation.country as country, COUNT(*) as cnt
FROM openalex.awards.mq_mental_health_awards
WHERE lead_investigator IS NOT NULL
GROUP BY 1 ORDER BY 2 DESC;

In [ ]:
%sql
SELECT COUNT(*) as mq_in_raw
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'mq_mental_health' AND priority = 271;